### **AgentLift: BT5153 Group 1**

---
### **Config & Imports**

In [1]:
import sys
import os
import json
import warnings
import pandas as pd
from pathlib import Path
from dataclasses import asdict
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

load_dotenv()

# ── CONFIG ───────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
AGENT_DIR      = Path(".")
MEMORY_PATH    = "logs/knowledge_base.jsonl"
LOGS_DIR       = Path("logs")
TRIALS_DIR     = Path("trials")
# ────────────────────────────────────────────────────────────────────────────

if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY not found.")

CURRENT_HYPOTHESIS = None  # Set to None on first run (cold start)

for d in [LOGS_DIR, TRIALS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(AGENT_DIR.resolve()))

from experiment_planning import (
    LLMClient,
    ExperimentMemory,
    CaseRetrievalAgent,
    HypothesisReasoningAgent,
    UpliftStrategySelectionAgent,
    TrialSpecWriterAgent,
    FeatureEngineeringExecutionAgent,
    TrialRecord,
)

llm    = LLMClient(provider="openai", api_key=OPENAI_API_KEY)
memory = ExperimentMemory(path=MEMORY_PATH)

print("✓ Config loaded")
print(f"  LLM          : {llm.provider} / {llm.model}")
print(f"  Memory path  : {MEMORY_PATH}")
stats = memory.summary_stats()
print(f"  Prior trials : {stats['total']} total, {stats['successful']} successful")
print(f"  Best AUUC    : {stats['best_auuc']}")

✓ Config loaded
  LLM          : openai / gpt-4o
  Memory path  : logs/knowledge_base.jsonl
  Prior trials : 0 total, 0 successful
  Best AUUC    : None


---
### **Load X5 Data**

In [2]:
pip install scikit-uplift

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sklift.datasets import fetch_x5

dataset      = fetch_x5()
clients_df   = dataset.data["clients"]
purchases_df = dataset.data["purchases"]
train_df     = pd.concat([
    dataset.data["train"],
    dataset.target.rename("target"),
    dataset.treatment.rename("treatment_flg"),
], axis=1).reset_index()
train_df.rename(columns={"index": "client_id"}, inplace=True)

print("✓ Data loaded via scikit-uplift fetch_x5()")
print(f"  clients_df   : {clients_df.shape}  | cols: {list(clients_df.columns)}")
print(f"  purchases_df : {purchases_df.shape} | cols: {list(purchases_df.columns)}")
print(f"  train_df     : {train_df.shape}     | cols: {list(train_df.columns)}")
print(f"\n  Treatment split:")
print(train_df["treatment_flg"].value_counts().to_string())
print(f"\n  Target split:")
print(train_df["target"].value_counts().to_string())

Part 1: X5 train:   0%|          | 0.00/1.18M [00:00<?, ?iB/s]

Part 2: X5 clients:   0%|          | 0.00/7.64M [00:00<?, ?iB/s]

Part 3: X5 purchases:   0%|          | 0.00/670M [00:00<?, ?iB/s]

✓ Data loaded via scikit-uplift fetch_x5()
  clients_df   : (400162, 5)  | cols: ['client_id', 'first_issue_date', 'first_redeem_date', 'age', 'gender']
  purchases_df : (45786568, 13) | cols: ['client_id', 'transaction_id', 'transaction_datetime', 'regular_points_received', 'express_points_received', 'regular_points_spent', 'express_points_spent', 'purchase_sum', 'store_id', 'product_id', 'product_quantity', 'trn_sum_from_iss', 'trn_sum_from_red']
  train_df     : (200039, 4)     | cols: ['client_id', 'client_id', 'target', 'treatment_flg']

  Treatment split:
treatment_flg
0    100058
1     99981

  Target split:
target
1    124002
0     76037


---
### **Phase II: Experiment Planning**

*Agent 1: Case Retrieval Agent*
> Reads experiment memory and surfaces prior evidence: similar recipes, supported/refuted hypotheses, best learner family, failed runs.

In [4]:
case_retrieval_agent = CaseRetrievalAgent(memory=memory, llm=llm)
retrieved_context    = case_retrieval_agent.run()

print("=" * 60)
print("CASE RETRIEVAL")
print("=" * 60)
print(f"Summary              : {retrieved_context.summary}")
print(f"Best learner family  : {retrieved_context.best_learner_family}")
print(f"Supported hypotheses : {retrieved_context.supported_hypotheses or 'None yet'}")
print(f"Refuted hypotheses   : {retrieved_context.refuted_hypotheses  or 'None yet'}")
print(f"Similar recipes      : {retrieved_context.similar_recipes     or 'None yet'}")
print(f"Failed runs          : {len(retrieved_context.failed_runs)} recorded")

CASE RETRIEVAL
Summary              : Cold start — no prior trial history found.
Best learner family  : SoloModel
Supported hypotheses : None yet
Refuted hypotheses   : None yet
Similar recipes      : None yet
Failed runs          : 0 recorded


*Agent 2: Hypothesis Reasoning Agent*
> Converts retrieved evidence + latest trial result into the next trial idea. Actions: **validate** | **refute** | **propose**.

In [5]:
hypothesis_agent = HypothesisReasoningAgent(llm=llm)
hypothesis = hypothesis_agent.run(
    context=retrieved_context,
    current_hypothesis=CURRENT_HYPOTHESIS,
    latest_trial=memory.best_trial("auuc"),
)

print("=" * 60)
print("HYPOTHESIS REASONING")
print("=" * 60)
print(f"Action     : {hypothesis.action.upper()}")
print(f"Hypothesis : {hypothesis.hypothesis}")
print(f"Evidence   : {hypothesis.evidence}")
print(f"Confidence : {hypothesis.confidence:.2f}")

HYPOTHESIS REASONING
Action     : PROPOSE
Hypothesis : When utilizing the SoloModel learner family, an uplift can be achieved through hyperparameter optimization focused on feature selection.
Evidence   : The context indicates a cold start with no prior trial history, suggesting an opportunity to explore initial uplift hypotheses.
Confidence : 0.20


*Agent 3: Uplift Strategy Selection Agent*
> Selects learner family (SoloModel / TwoModels / ResponseModel / ClassTransformation), base estimator, feature recipe, split seed, and evaluation cutoff.

In [6]:
strategy_agent     = UpliftStrategySelectionAgent(memory=memory, llm=llm)
strategy           = strategy_agent.run(hypothesis=hypothesis, context=retrieved_context)
estimator_defaults = strategy_agent.get_estimator_params(strategy.base_estimator)

print("=" * 60)
print("UPLIFT STRATEGY SELECTION")
print("=" * 60)
print(f"Learner family   : {strategy.learner_family}")
print(f"Base estimator   : {strategy.base_estimator}")
print(f"Feature recipe   : {strategy.feature_recipe}")
print(f"Split seed       : {strategy.split_seed}")
print(f"Eval cutoff      : {strategy.eval_cutoff}")
print(f"Rationale        : {strategy.rationale}")
print(f"\nDefault params for {strategy.base_estimator}:")
print(json.dumps(estimator_defaults, indent=2))

UPLIFT STRATEGY SELECTION
Learner family   : ResponseModel
Base estimator   : XGBoost
Feature recipe   : rfm_baseline
Split seed       : 42
Eval cutoff      : 0.3
Rationale        : Warm-up trial for ResponseModel with default params.

Default params for XGBoost:
{
  "n_estimators": 100,
  "max_depth": 5,
  "learning_rate": 0.1,
  "use_label_encoder": false,
  "eval_metric": "logloss"
}


*Agent 4: Trial Spec Writer Agent*
> Produces a fully resolved trial plan: hypothesis + what changed + expected improvement + exact model & params + feature recipe + stop criteria.

In [7]:
trial_spec_agent = TrialSpecWriterAgent(memory=memory, llm=llm)
trial_spec = trial_spec_agent.run(
    hypothesis=hypothesis,
    strategy=strategy,
    estimator_defaults=estimator_defaults,
)

print("=" * 60)
print("TRIAL SPEC")
print("=" * 60)
print(json.dumps(asdict(trial_spec), indent=2))

# Save trial spec for Phase III handoff
trial_spec_path = LOGS_DIR / "trial_spec.json"
with open(trial_spec_path, "w") as f:
    json.dump(asdict(trial_spec), f, indent=2)
print(f"\n✓ Trial spec saved → {trial_spec_path}")

TRIAL SPEC
{
  "trial_id": "d9f5d4b9-3261-4b5f-b9dd-4b8c3e34e3ed",
  "hypothesis": "Utilizing the TwoModels approach with LightGBM can lead to better uplift due to earlier separation of treatment and control modeling.",
  "changes_from_previous": "Switched from SoloModel with default parameters to TwoModels using LightGBM.",
  "expected_improvement": "AUUC +0.01 over SoloModel baseline",
  "model": "TwoModels + LightGBM",
  "params": {
    "n_estimators": 150,
    "max_depth": 7,
    "learning_rate": 0.05,
    "num_leaves": 31
  },
  "feature_recipe": "rfm_baseline",
  "stop_criteria": "Automatic if AUUC does not improve after 3 continuous trials"
}

✓ Trial spec saved → logs\trial_spec.json


*Agent 5: Feature Engineering Execution Agent*
> Builds the feature table requested by the Trial Spec Writer Agent. Validates one row per customer_id, no leakage, all train customers present.

In [8]:
feature_agent = FeatureEngineeringExecutionAgent(
    clients_df=clients_df,
    purchases_df=purchases_df,
    train_df=train_df,
)

feature_table = feature_agent.run(
    trial_spec=trial_spec,
    context=retrieved_context,
)

print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)
print(f"Recipe           : {feature_table.recipe_name}")
print(f"Customers        : {feature_table.n_customers}")
print(f"Features built   : {len(feature_table.feature_names)}")
print(f"Validation passed: {feature_table.validation_passed}")
print(f"Leakage check    : {feature_table.leakage_check_passed}")
print(f"Skipped families : {feature_table.skipped_families or 'None'}")

print("\nValidation notes:")
for note in feature_table.validation_notes:
    print(f"  {note}")

print("\nFeature names:")
print(feature_table.feature_names)

print("\nSample (5 rows):")
display(feature_table.feature_df.head())

FEATURE ENGINEERING
Recipe           : rfm_baseline
Customers        : 400162
Features built   : 3
Validation passed: True
Leakage check    : True
Skipped families : None

Validation notes:
  All checks passed.

Feature names:
['recency_days', 'frequency', 'monetary_total']

Sample (5 rows):


,customer_id,recency_days,frequency,monetary_total
0,000012768d,4,52,40809.00
1,000036f903,1,162,58765.00
2,000048b7a6,6,56,29724.00
3,000073194a,2,82,62719.92
4,00007c7133,14,83,53998.72


*Validation Gate*
> Hard stop if feature table has leakage or duplicate customer_ids. Pipeline will not proceed to Phase III with a bad feature table.

In [9]:
# ── Validation gate — do not pass a bad feature table to Phase III ───────────
assert feature_table.leakage_check_passed, (
    "PIPELINE HALTED: leakage detected in feature table. "
    "Check feature_table.validation_notes for details."
)
assert feature_table.validation_passed, (
    "PIPELINE HALTED: feature table failed validation (likely duplicate customer_ids). "
    "Check feature_table.validation_notes for details."
)

print("✓ Validation gate passed — feature table is clean")

✓ Validation gate passed — feature table is clean


*Output (Save Handoff Files)*
> Saves all outputs Phase III and Phase V need

In [11]:
from datetime import datetime

# ── 1. Feature table → parquet (consumed by Phase III training agents) ────────
feature_table_path = LOGS_DIR / "feature_table.parquet"
feature_table.feature_df.to_parquet(feature_table_path, index=False)
print(f"✓ Feature table saved  → {feature_table_path}")

# ── 2. Trial spec → JSON (already saved in Cell 6, confirm here) ──────────────
print(f"✓ Trial spec confirmed → {LOGS_DIR / 'trial_spec.json'}")

# ── 3. Trial meta → trials/{trial_id}/trial_meta.json (Sherlyn's format) ─────
trial_output_dir = TRIALS_DIR / trial_spec.trial_id
trial_output_dir.mkdir(parents=True, exist_ok=True)

trial_meta = {
    "trial_id":        trial_spec.trial_id,
    "parent_run_id":   None,
    "hypothesis_id":   f"H_{trial_spec.trial_id[:6]}",
    "hypothesis_text": trial_spec.hypothesis,
    "model_type":      strategy.learner_family,
    "base_estimator":  strategy.base_estimator,
    "feature_recipe":  feature_table.feature_names,
    "params":          trial_spec.params,
    "split_seed":      strategy.split_seed,
    "dataset_version": "x5_retailhero_v1",
    "phase2_timestamp": datetime.now().isoformat(),
    "stop_criteria":   trial_spec.stop_criteria,
    "changes_from_previous": trial_spec.changes_from_previous,
    "expected_improvement":  trial_spec.expected_improvement,
}

meta_path = trial_output_dir / "trial_meta.json"
with open(meta_path, "w") as f:
    json.dump(trial_meta, f, indent=2)
print(f"✓ Trial meta saved     → {meta_path}")

# ── 4. Phase II summary ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE II COMPLETE — HANDOFF SUMMARY")
print("=" * 60)
print(f"Trial ID         : {trial_spec.trial_id}")
print(f"Hypothesis       : {trial_spec.hypothesis}")
print(f"Action           : {hypothesis.action.upper()}")
print(f"Model            : {trial_spec.model}")
print(f"Feature recipe   : {trial_spec.feature_recipe} ({len(feature_table.feature_names)} features)")
print(f"Stop criteria    : {trial_spec.stop_criteria}")
print()
print("Files ready for Phase III")
print(f"  → {feature_table_path}")
print(f"  → {LOGS_DIR / 'trial_spec.json'}")
print()
print("Files ready for Phase V")
print(f"  → {meta_path}")

✓ Feature table saved  → logs\feature_table.parquet
✓ Trial spec confirmed → logs\trial_spec.json
✓ Trial meta saved     → trials\d9f5d4b9-3261-4b5f-b9dd-4b8c3e34e3ed\trial_meta.json

PHASE II COMPLETE — HANDOFF SUMMARY
Trial ID         : d9f5d4b9-3261-4b5f-b9dd-4b8c3e34e3ed
Hypothesis       : Utilizing the TwoModels approach with LightGBM can lead to better uplift due to earlier separation of treatment and control modeling.
Action           : PROPOSE
Model            : TwoModels + LightGBM
Feature recipe   : rfm_baseline (3 features)
Stop criteria    : Automatic if AUUC does not improve after 3 continuous trials

Files ready for Phase III
  → logs\feature_table.parquet
  → logs\trial_spec.json

Files ready for Phase V
  → trials\d9f5d4b9-3261-4b5f-b9dd-4b8c3e34e3ed\trial_meta.json
